In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver      
from selenium.webdriver.common.by import By
import time 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import random
from IPython.display import display

In [2]:
'''url = 'https://www.dges.gov.pt/guias/indcurso.asp'
browser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options
browser.get(url)  # Open the URL in the browser
time.sleep(1)  '''

"url = 'https://www.dges.gov.pt/guias/indcurso.asp'\nbrowser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options\nbrowser.get(url)  # Open the URL in the browser\ntime.sleep(1)  "

In [3]:
'''def scrape_current_letter(browser, course_institution_data):
    """Scrape all courses & institutions from the current letter page."""
    WebDriverWait(browser, 20).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
    )

    soup = BeautifulSoup(browser.page_source, "html.parser")
    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])
    current_course = None

    for block in all_blocks:
        classes = block.get("class", [])

        # --- Course block ---
        if "box10" in classes:
            name_tag = block.find("div", class_="lin-area-c2")
            if name_tag:
                current_course = name_tag.text.strip()
                print(f"\n📘 Course: {current_course}")

        # --- Institution block ---
        elif "lin-curso" in classes and current_course:
            link_tag = block.find("a")
            if not link_tag:
                continue

            institution = link_tag.text.strip()
            href = link_tag.get("href")

            if any(kw in institution for kw in ["Universidade", "Instituto", "Escola", "Politécnico"]):
                print(f"🏫 Institution: {institution}")

                try:
                    # Click the institution
                    inst_element = WebDriverWait(browser, 10).until(
                        EC.element_to_be_clickable((By.XPATH, f"//a[@href='{href}']"))
                    )
                    browser.execute_script("arguments[0].scrollIntoView(true);", inst_element)
                    time.sleep(0.3)
                    inst_element.click()

                    # Wait for detail page
                    WebDriverWait(browser, 15).until(
                        EC.presence_of_all_elements_located((By.CLASS_NAME, "inside2"))
                    )
                    time.sleep(1)

                    # Parse the detail page
                    detail_soup = BeautifulSoup(browser.page_source, "html.parser")

                    # --- Extract Google Maps link ---
                    google_map = ""
                    inside_block = detail_soup.find("div", class_="inside2")
                    if inside_block:
                        map_link = inside_block.find("a", href=True, string=lambda t: t and "Mapa" in t)
                        if not map_link:
                            map_span = inside_block.find("span", class_="vislink", string=lambda t: "Mapa" in t)
                            if map_span and map_span.parent.name == "a":
                                map_link = map_span.parent
                        if map_link:
                            google_map = map_link["href"].strip()

                    print(f"🗺️ Google Maps link: {google_map if google_map else 'Not found'}")

                except Exception as e:
                    print(f"⚠️ Error scraping {institution}: {e}")
                    google_map = ""

                # Go back to the list page
                browser.back()
                time.sleep(1)
                WebDriverWait(browser, 20).until(
                    EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
                )

                # Store
                course_institution_data.append((current_course, institution, href, google_map))

    return course_institution_data


# --- Step 1: Scrape letter A (already selected) ---
print("\n🔤 Scraping letter: A (default)")
course_institution_data = []
course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Step 2: Click and scrape all other letters ---
letters = browser.find_elements(By.CSS_SELECTOR, "div.noprint a")
letter_links = [(a.text.strip(), a.get_attribute("href")) for a in letters if a.text.strip()]

for letter, link in letter_links:
    print(f"\n🔤 Scraping letter: {letter}")
    browser.get(link)
    time.sleep(1.5)
    course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Save results ---
df = pd.DataFrame(course_institution_data, columns=["Course", "Institution", "Link", "GoogleMaps"])'''

'def scrape_current_letter(browser, course_institution_data):\n    """Scrape all courses & institutions from the current letter page."""\n    WebDriverWait(browser, 20).until(\n        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))\n    )\n\n    soup = BeautifulSoup(browser.page_source, "html.parser")\n    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])\n    current_course = None\n\n    for block in all_blocks:\n        classes = block.get("class", [])\n\n        # --- Course block ---\n        if "box10" in classes:\n            name_tag = block.find("div", class_="lin-area-c2")\n            if name_tag:\n                current_course = name_tag.text.strip()\n                print(f"\n📘 Course: {current_course}")\n\n        # --- Institution block ---\n        elif "lin-curso" in classes and current_course:\n            link_tag = block.find("a")\n            if not link_tag:\n                continue\n\n            institution = link_tag.text.strip(

In [4]:
# --------------------- Chrome setup ---------------------
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# --------------------- Open search page ---------------------
driver.get("https://www.mastersportal.com/search/master/portugal")

# Accept cookies if popup appears
try:
    cookie_btn = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Accept')]"))
    )
    cookie_btn.click()
    print("Cookies accepted.")
except:
    print("No cookie popup detected.")

# --------------------- Lists to store data ---------------------
names = []
universities = []
locations = []
durations = []
tuitions = []
abouts = []

seen = set()  # to avoid duplicates

# --------------------- Main scraping loop ---------------------
while True:
    print("\n=== Scraping new page ===")

    # Wait until at least one card is loaded
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))

    # Scroll to load all cards
    previous_len = 0
    same_count_rounds = 0
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
        new_len = len(cards)

        if new_len == previous_len:
            same_count_rounds += 1
        else:
            same_count_rounds = 0

        if same_count_rounds >= 3:
            break
        previous_len = new_len

    # Final scroll
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
    print(f"Found {len(cards)} cards on this page.")

    # --------------------- Extract data from each card ---------------------
    for card in cards:
        try:
            name = card.find_element(By.CSS_SELECTOR, "h2.StudyName").text
        except:
            name = ""
        try:
            uni = card.find_element(By.CSS_SELECTOR, "strong.OrganisationName").text
        except:
            uni = ""
        key = (name, uni)
        if key in seen or name == "":
            continue
        seen.add(key)

        print(f"Processing Master: {name} | University: {uni}")

        try:
            loc = card.find_element(By.CSS_SELECTOR, "strong.OrganisationLocation").text
        except:
            loc = ""
        try:
            duration = card.find_element(By.CSS_SELECTOR, ".DurationValue").text
        except:
            duration = ""
        try:
            tuition = card.find_element(By.CSS_SELECTOR, ".TuitionValue").text
        except:
            tuition = ""

        # --------------------- Click button to load About ---------------------
        try:
            view_btn = card.find_element(By.CSS_SELECTOR, "div.StudyCardFooter .CTA")
            driver.execute_script(
                "arguments[0].scrollIntoView({behavior:'smooth', block:'center'});", view_btn
            )
            time.sleep(0.5)
            view_btn.click()

            # Wait until About section appears
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "section#StudySummary p")))
            time.sleep(1)  # allow content to render

            about_section = driver.find_element(By.CSS_SELECTOR, "section#StudySummary")
            paragraphs = about_section.find_elements(By.TAG_NAME, "p")
            about = paragraphs[0].text if paragraphs else ""
            print("About extracted successfully.")

            # Go back to search page
            driver.back()
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))
            time.sleep(1)
        except Exception as e:
            print("Error extracting About:", e)
            about = ""

        # Append all data
        names.append(name)
        universities.append(uni)
        locations.append(loc)
        durations.append(duration)
        tuitions.append(tuition)
        abouts.append(about)

        print(f"Total cards processed: {len(names)}\n")
        time.sleep(random.uniform(1.0, 2.0))

    # --------------------- Pagination ---------------------
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "a.NextButton")
        if "disabled" in next_btn.get_attribute("class"):
            print("Reached last page.")
            break
        else:
            driver.execute_script(
                "arguments[0].scrollIntoView({behavior:'smooth', block:'center'});", next_btn
            )
            time.sleep(1)
            next_btn.click()
            print("Moving to next page...")
            time.sleep(3)  # wait for new page
    except:
        print("No Next button found, finished scraping.")
        break

No cookie popup detected.

=== Scraping new page ===
Found 20 cards on this page.
Processing Master: Sustainable Urban Mobility Transitions | University: EIT Urban Mobility Master School
Error extracting About: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff6b1d3a235
	0x7ff6b1a92630
	0x7ff6b18216dd
	0x7ff6b187a27e
	0x7ff6b187a58c
	0x7ff6b18ced77
	0x7ff6b18cbaba
	0x7ff6b186b0ed
	0x7ff6b186bf63
	0x7ff6b1d65d60
	0x7ff6b1d5fe8a
	0x7ff6b1d81005
	0x7ff6b1aad71e
	0x7ff6b1ab4e1f
	0x7ff6b1a9b7c4
	0x7ff6b1a9b97f
	0x7ff6b1a818e8
	0x7ff85da0e8d7
	0x7ff85e90c53c

Total cards processed: 1

Processing Master: Smart Mobility Data Science and Analytics | University: EIT Urban Mobility Master School
Error extracting About: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff6b1d3a235
	0x7ff6b1a92630
	0x7ff6b18216dd
	0x7ff6b187a27e
	0x7ff6b187a58c
	0x7ff6b18ced77
	0x7ff6b18cbaba
	0x7ff6b186b0ed
	0x7ff6b186bf63
	0x7ff6b1d65d60
	0x7ff6b1d5fe8a


In [ ]:
# --------------------- Chrome setup ---------------------
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# --------------------- Open search page ---------------------
driver.get("https://www.mastersportal.com/search/master/portugal")

# Accept cookies if popup appears
try:
    cookie_btn = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Accept')]"))
    )
    cookie_btn.click()
    print("Cookies accepted.")
except:
    print("No cookie popup detected.")

# --------------------- Lists to store data ---------------------
names = []
universities = []
locations = []
durations = []
tuitions = []
abouts = []

seen = set()  # to avoid duplicates

# --------------------- Loop over pages ---------------------
while True:
    # Wait until at least one card is loaded
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))

    # Scroll to load all cards
    previous_len = 0
    same_count_rounds = 0
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
        new_len = len(cards)

        if new_len == previous_len:
            same_count_rounds += 1
        else:
            same_count_rounds = 0

        if same_count_rounds >= 3:
            break
        previous_len = new_len

    # Final scroll
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
    print(f"Found {len(cards)} cards on this page.")

    # --------------------- Extract data ---------------------
    for card in cards:
        try:
            name = card.find_element(By.CSS_SELECTOR, "h2.StudyName").text
        except:
            name = ""
        try:
            uni = card.find_element(By.CSS_SELECTOR, "strong.OrganisationName").text
        except:
            uni = ""
        key = (name, uni)
        if key in seen or name == "":
            continue
        seen.add(key)

        print(f"Processing Master: {name} | University: {uni}")

        try:
            loc = card.find_element(By.CSS_SELECTOR, "strong.OrganisationLocation").text
        except:
            loc = ""
        try:
            duration = card.find_element(By.CSS_SELECTOR, ".DurationValue").text
        except:
            duration = ""
        try:
            tuition = card.find_element(By.CSS_SELECTOR, ".TuitionValue").text
        except:
            tuition = ""

        # --------------------- Go to program page for About ---------------------
        try:
            program_link = card.get_attribute("href")
            driver.get(program_link)

            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "section#StudySummary")))
            time.sleep(1)  # allow About section to populate

            about_section = driver.find_element(By.CSS_SELECTOR, "section#StudySummary")
            about = about_section.find_element(By.TAG_NAME, "p").text
            print("About extracted successfully.")

            # Go back to search results
            driver.back()
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))
            time.sleep(1)
        except Exception as e:
            print("Error extracting About:", e)
            about = ""

        # Append data
        names.append(name)
        universities.append(uni)
        locations.append(loc)
        durations.append(duration)
        tuitions.append(tuition)
        abouts.append(about)

        # Debug: number of cards processed
        print(f"Total cards processed: {len(names)}\n")
        time.sleep(random.uniform(1.0, 2.5))  # mimic human behavior

    # --------------------- Pagination ---------------------
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "a.NextButton")
        if "disabled" in next_btn.get_attribute("class"):
            print("Reached last page.")
            break
        else:
            next_btn.click()
            time.sleep(3)
    except:
        print("No Next button found, finished scraping.")
        break

# --------------------- Save DataFrame ---------------------
df = pd.DataFrame({
    "Master Name": names,
    "University": universities,
    "Location": locations,
    "Duration": durations,
    "Tuition Fee": tuitions,
    "About": abouts
})

# Save to CSV
df.to_csv("masters_portugal.csv", index=False)
print("✅ Scraping finished, data saved to masters_portugal.csv")


In [5]:
# Create DataFrame
df = pd.DataFrame({
    "Master Name": names,
    "University": universities,
    "Location": locations,
    "Duration": durations,
    "Tuition Fee": tuitions,
    "About": abouts
})
df

,Master Name,University,Location,Duration,Tuition Fee,About
0,Sustainable Urban Mobility Transitions,EIT Urban Mobility Master School,Multiple locations,2 years,4000 EUR / year,
1,Smart Mobility Data Science and Analytics,EIT Urban Mobility Master School,Multiple locations,2 years,4000 EUR / year,
2,Postgraduate Program in Artificial Intelligenc...,NOVA IMS,"Lisbon, Portugal",9 months,4100 EUR / year,
3,Postgraduate Program in Enterprise Data Scienc...,NOVA IMS,"Lisbon, Portugal",9 months,5100 EUR / year,


In [6]:
df.to_csv("masters_portugal_all_pages.csv", index=False)